In [1]:
import os
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'GPUs Available: {torch.cuda.device_count()}')

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

Device: cuda
GPU: NVIDIA H200
VRAM: 150.1 GB
GPUs Available: 6


In [2]:
CONFIG = {
    'data_root': '/nfsshare/users/P126157010/Genimage/Genimage',

    'generators': [
        'BigGan/imagenet_ai_0419_biggan',
        'Glide/imagenet_glide',
        'vqdm/imagenet_ai_0419_vqdm',
        'stable_diffusion/imagenet_ai_0419_sdv4'
    ],
    'image_size': 299,
    'batch_size': 128,
    'num_workers': 16,
    'pin_memory': True,
    'prefetch_factor': 4,
    'learning_rate': 0.001,
    'num_epochs': 20,      
    'weight_decay': 1e-4,
    'momentum': 0.9,
    'num_classes': 2,
    'device': device,
    'results_dir': './results',
    'checkpoint_dir': './checkpoints',
    'use_amp': True,
}
os.makedirs(CONFIG['results_dir'], exist_ok=True)
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)

In [3]:
class AIGeneratorDataset(Dataset):
    """
    Custom dataset for per-generator loading.
    Labels: 0 = real (nature), 1 = AI-generated (ai)
    """

    def __init__(self, root: str, generator: str, split: str, transform=None):
        self.transform = transform
        self.samples: List[Tuple[str, int]] = []

        gen_dir = os.path.join(root, generator, split)

        class_map = {
            'nature': 0,
            'ai': 1
        }

        for cls, label in class_map.items():
            cls_dir = os.path.join(gen_dir, cls)

            if not os.path.isdir(cls_dir):
                raise FileNotFoundError(f'Directory not found: {cls_dir}')

            for fname in os.scandir(cls_dir):
                if fname.is_file() and fname.name.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                    self.samples.append((fname.path, label))

        print(f'[{generator}/{split}] {len(self.samples)} images | '
              f'real={sum(1 for _,l in self.samples if l==0)}, '
              f'fake={sum(1 for _,l in self.samples if l==1)}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, label = self.samples[idx]

        try:
            img = Image.open(path).convert('RGB')
        except:
            idx = (idx + 1) % len(self.samples)
            path, label = self.samples[idx]
            img = Image.open(path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img, label

In [4]:
class FullDataset(Dataset):
    def __init__(self, root: str, generators: List[str], split: str, transform=None):
        self.transform = transform
        self.samples: List[Tuple[str, int]] = []

        class_map = {
            'nature': 0,
            'ai': 1
        }

        for gen in generators:
            gen_dir = os.path.join(root, gen, split)

            for cls, label in class_map.items():
                cls_dir = os.path.join(gen_dir, cls)

                if not os.path.isdir(cls_dir):
                    continue

                for fname in os.scandir(cls_dir):
                    if fname.is_file() and fname.name.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                        self.samples.append((fname.path, label))

        print(f'[Full/{split}] {len(self.samples)} total images')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, label = self.samples[idx]

        try:
            img = Image.open(path).convert('RGB')
        except:
            idx = (idx + 1) % len(self.samples)
            path, label = self.samples[idx]
            img = Image.open(path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img, label


In [5]:
def get_transforms(split: str) -> transforms.Compose:
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

In [6]:
def make_loader(dataset: Dataset, shuffle: bool = True, drop_last=True) -> DataLoader:
    loader_kwargs = dict(
        dataset=dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=shuffle,
        num_workers=CONFIG['num_workers'],
        pin_memory=CONFIG['pin_memory'],
        drop_last=drop_last,
    )

    if CONFIG['num_workers'] > 0:
        loader_kwargs['prefetch_factor'] = CONFIG['prefetch_factor']
        loader_kwargs['persistent_workers'] = True

    return DataLoader(**loader_kwargs)

In [7]:
class ExpertModel(nn.Module):
    """
    ResNet-50 expert specialised for one generator type.
    Outputs 2 logits (real vs fake).
    """

    def __init__(self, generator_name: str):
        super().__init__()
        self.generator_name = generator_name

        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.feature_dim = backbone.fc.in_features  # 2048

        self.backbone = nn.Sequential(*list(backbone.children())[:-1])

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(self.feature_dim, 2)
        )

        nn.init.xavier_uniform_(self.classifier[1].weight)
        nn.init.zeros_(self.classifier[1].bias)

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.backbone(x)
        return feats.flatten(1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.forward_features(x)
        return self.classifier(feats)  
def build_expert(generator_name: str) -> ExpertModel:
    model = ExpertModel(generator_name).to(device)
    model = model.to(memory_format=torch.channels_last)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'Expert [{generator_name}]: {n_params:.1f}M params')
    return model

In [8]:
class GatingNetwork(nn.Module):
    def __init__(self, n_experts: int = 4, hidden_dim: int = 16):
        super().__init__()
        # Raw image flattened: 3 × 299 × 299 = 268,203
        input_dim = 3 * CONFIG['image_size'] * CONFIG['image_size']

        self.gate = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, n_experts),
            nn.Softmax(dim=1)
        )
        nn.init.xavier_uniform_(self.gate[0].weight)
        nn.init.zeros_(self.gate[0].bias)
        nn.init.xavier_uniform_(self.gate[2].weight)
        nn.init.zeros_(self.gate[2].bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.gate(x.flatten(1))   # (B, 268203) → (B, 4)


class GatedMoE(nn.Module):
    def __init__(self, gating_net: GatingNetwork, experts: Dict[str, ExpertModel]):
        super().__init__()
        self.gating_net   = gating_net
        self.experts      = nn.ModuleDict(experts)
        self.expert_names = list(experts.keys())

    def forward(self, x: torch.Tensor):
        gate_weights = self.gating_net(x)                              # (B, 4)
        all_logits   = [self.experts[name](x) for name in self.expert_names]
        stacked      = torch.stack(all_logits, dim=1)                  # (B, 4, 2)
        weighted     = torch.sum(gate_weights.unsqueeze(-1) * stacked, dim=1)  # (B, 2)
        return weighted, gate_weights

In [9]:
class AttentionGatingNetwork(nn.Module):
    """
    Patch-based attention gating:
    1. Splits image into patches → patch tokens
    2. Expert logits → expert tokens
    3. Cross-attends expert tokens (query) over patch tokens (key/value)
    4. Self-attends among experts
    5. Produces dynamic per-expert softmax weights
    """
    def __init__(self, n_experts: int = 4, patch_size: int = 23,
                 embed_dim: int = 128, num_heads: int = 4,
                 ff_dim: int = 256, dropout: float = 0.1):
        super().__init__()
        self.patch_size = patch_size
        # image_size=299, patch_size=23 → 13x13 = 169 patches
        n_patches = (CONFIG['image_size'] // patch_size) ** 2
        patch_dim = 3 * patch_size * patch_size   # raw patch pixels

        # Patch embedding
        self.patch_embed = nn.Sequential(
            nn.Linear(patch_dim, embed_dim),
            nn.LayerNorm(embed_dim),
        )

        # Expert logit embedding: 2 logits per expert → embed_dim
        self.expert_embed = nn.Linear(2, embed_dim)

        # Cross-attention: expert tokens query patch tokens
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=embed_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)

        # Self-attention among expert tokens
        self.self_attn = nn.MultiheadAttention(
            embed_dim=embed_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim),
        )
        self.norm3     = nn.LayerNorm(embed_dim)
        self.gate_head = nn.Linear(embed_dim, 1)
        self.softmax   = nn.Softmax(dim=1)

    def _patchify(self, x: torch.Tensor) -> torch.Tensor:
        """Split (B,3,H,W) into (B, n_patches, patch_dim)"""
        B, C, H, W = x.shape
        p = self.patch_size
        x = x.unfold(2, p, p).unfold(3, p, p)        # (B,C,nh,nw,p,p)
        x = x.contiguous().view(B, C, -1, p, p)       # (B,C,n_patches,p,p)
        x = x.permute(0, 2, 1, 3, 4)                  # (B,n_patches,C,p,p)
        return x.contiguous().view(B, x.shape[1], -1) # (B,n_patches,C*p*p)

    def forward(self, x: torch.Tensor,
                expert_logits: torch.Tensor) -> torch.Tensor:
        """
        x:             (B, 3, H, W)      raw image
        expert_logits: (B, n_experts, 2) each expert's real/fake logits
        returns:       (B, n_experts)    softmax gate weights
        """
        # 1. Patch tokens from raw image
        patches      = self._patchify(x)                     # (B, n_patches, patch_dim)
        patch_tokens = self.patch_embed(patches)             # (B, n_patches, embed_dim)

        # 2. Expert tokens from their logits
        expert_tokens = self.expert_embed(expert_logits)     # (B, n_experts, embed_dim)

        # 3. Cross-attention: experts query the image patches
        attn_out, _ = self.cross_attn(
            query=expert_tokens,
            key=patch_tokens,
            value=patch_tokens)
        expert_tokens = self.norm1(expert_tokens + attn_out) # (B, n_experts, embed_dim)

        # 4. Self-attention among expert tokens
        sa_out, _     = self.self_attn(expert_tokens, expert_tokens, expert_tokens)
        expert_tokens = self.norm2(expert_tokens + sa_out)

        # 5. FF + gate head → softmax weights
        expert_tokens = self.norm3(expert_tokens + self.ff(expert_tokens))
        return self.softmax(self.gate_head(expert_tokens).squeeze(-1))  # (B, n_experts)


class AttentionGatedMoE(nn.Module):
    def __init__(self, attn_gate: AttentionGatingNetwork, experts: Dict[str, ExpertModel]):
        super().__init__()
        self.attn_gate    = attn_gate
        self.experts      = nn.ModuleDict(experts)
        self.expert_names = list(experts.keys())
        # Freeze all expert parameters — only attention gate trains
        for param in self.experts.parameters():
            param.requires_grad = False

    def forward(self, x: torch.Tensor):
        with torch.no_grad():
            logit_list = [self.experts[name](x) for name in self.expert_names]

        stacked_logits = torch.stack(logit_list, dim=1)           # (B, n_experts, 2)

        # Gate sees raw image patches AND expert logits
        gate_weights = self.attn_gate(x, stacked_logits)          # (B, n_experts)

        weighted = torch.sum(
            gate_weights.unsqueeze(-1) * stacked_logits, dim=1)   # (B, 2)
        return weighted, gate_weights


In [10]:
def make_optimizer(model: nn.Module) -> optim.SGD:
    return optim.SGD(
        model.parameters(),
        lr=CONFIG['learning_rate'],
        momentum=CONFIG['momentum'],
        weight_decay=CONFIG['weight_decay'],
        nesterov=True,
    )

In [11]:
def make_scheduler(optimizer: optim.Optimizer, loader: DataLoader):
    return optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=CONFIG['learning_rate'] * 10,
        epochs=CONFIG['num_epochs'],
        steps_per_epoch=len(loader),
        pct_start=0.3,
    )

In [12]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
    }

In [13]:
def print_metrics(metrics: Dict[str, float], prefix: str = '') -> None:
    print(f'{prefix}  Acc={metrics["accuracy"]:.4f}  '
          f'Prec={metrics["precision"]:.4f}  '
          f'Rec={metrics["recall"]:.4f}  '
          f'F1={metrics["f1"]:.4f}')


@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader):
    model.eval()
    y_true, y_pred = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        logits = model(images)
        preds  = torch.argmax(logits, dim=1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(labels.numpy())

    return compute_metrics(np.array(y_true), np.array(y_pred))

In [14]:
class EarlyStopping:
    def __init__(self, patience: int = 7, min_delta: float = 1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = None
        self.counter    = 0
        self.should_stop = False

    def step(self, score: float) -> bool:
        if self.best_score is None or score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop

In [15]:
def make_history():
    return {
        'train_loss': [],
        'val_acc': [], 'val_prec': [], 'val_rec': [], 'val_f1': [],
        'lr': [],
        'early_stop_counter': [],
    }

In [16]:
def train_expert(model, train_loader, val_loader, generator_name):
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(model)
    scheduler = make_scheduler(optimizer, train_loader)
    scaler    = GradScaler('cuda', enabled=CONFIG['use_amp'])
    stopper   = EarlyStopping(patience=5)
    history   = make_history()
    best_f1   = 0.0

    for epoch in range(CONFIG['num_epochs']):
        model.train()
        running_loss = 0.0
        t0 = time.time()

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.long().to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast('cuda', enabled=CONFIG['use_amp']):
                logits = model(images)
                loss   = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        metrics  = evaluate_model(model, val_loader)
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(avg_loss)
        history['val_acc'].append(metrics['accuracy'])
        history['val_prec'].append(metrics['precision'])
        history['val_rec'].append(metrics['recall'])
        history['val_f1'].append(metrics['f1'])
        history['lr'].append(current_lr)
        history['early_stop_counter'].append(stopper.counter)

        elapsed = time.time() - t0
        print(f"[{generator_name}] Epoch {epoch+1:02d}/{CONFIG['num_epochs']} "
              f"loss={avg_loss:.4f} lr={current_lr:.6f} "
              f"ES={stopper.counter}/{stopper.patience} {elapsed:.1f}s", end=' ')
        print_metrics(metrics)

        if metrics['f1'] > best_f1:
            best_f1 = metrics['f1']
            safe_name = generator_name.replace('/', '_')
            torch.save(model.state_dict(),
                os.path.join(CONFIG['checkpoint_dir'], f'expert_{safe_name}_best.pt'))
            
        if stopper.step(metrics['f1']):
            print(f"  [Early Stop] Triggered at epoch {epoch+1} "
                  f"(no improvement for {stopper.patience} epochs)")
            break

    safe_name = generator_name.replace('/', '_')
    model.load_state_dict(torch.load(
        os.path.join(CONFIG['checkpoint_dir'], f'expert_{safe_name}_best.pt'),
        map_location=device))
    print(f"[{generator_name}] Best F1: {best_f1:.4f}")
    return history


In [17]:
def train_gated_model(gated_model, train_loader, val_loader,
                      ckpt_name: str = 'gated_best.pt'):
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(gated_model)
    scheduler = make_scheduler(optimizer, train_loader)
    scaler    = GradScaler('cuda', enabled=CONFIG['use_amp'])
    stopper   = EarlyStopping(patience=5)
    history   = make_history()
    best_f1   = 0.0
    ckpt_path = os.path.join(CONFIG['checkpoint_dir'], ckpt_name)

    for epoch in range(CONFIG['num_epochs']):
        gated_model.train()
        running_loss = 0.0
        t0 = time.time()

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.long().to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast('cuda', enabled=CONFIG['use_amp']):
                logits, _ = gated_model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(gated_model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += loss.item()

        avg_loss   = running_loss / len(train_loader)
        metrics    = evaluate_gated_moe(gated_model, val_loader)
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(avg_loss)
        history['val_acc'].append(metrics['accuracy'])
        history['val_prec'].append(metrics['precision'])
        history['val_rec'].append(metrics['recall'])
        history['val_f1'].append(metrics['f1'])
        history['lr'].append(current_lr)
        history['early_stop_counter'].append(stopper.counter)

        elapsed = time.time() - t0
        print(f"[Gated] Epoch {epoch+1:02d}/{CONFIG['num_epochs']} "
              f"loss={avg_loss:.4f} lr={current_lr:.6f} "
              f"ES={stopper.counter}/{stopper.patience} {elapsed:.1f}s", end=' ')
        print_metrics(metrics)

        if metrics['f1'] > best_f1:
            best_f1 = metrics['f1']
            torch.save(gated_model.state_dict(), ckpt_path)

        if stopper.step(metrics['f1']):
            print(f"  [Early Stop] Triggered at epoch {epoch+1} "
                  f"(no improvement for {stopper.patience} epochs)")
            break

    gated_model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f"[Gated] Best F1: {best_f1:.4f}")
    return history


In [18]:
@torch.no_grad()
def evaluate_gating(
    gating_net: GatingNetwork,
    frozen_experts: Dict[str, ExpertModel],
    loader: DataLoader,
) -> Dict[str, float]:
    gating_net.eval()
    for m in frozen_experts.values():
        m.eval()
    expert_names   = list(frozen_experts.keys())
    y_true, y_pred = [], []
    for images, labels in loader:
        images       = images.to(device, non_blocking=True)
        gate_weights = gating_net(images)
        all_logits   = [frozen_experts[name](images) for name in expert_names]
        stacked      = torch.stack(all_logits, dim=1)
        weighted     = torch.sum(gate_weights.unsqueeze(-1) * stacked, dim=1)
        preds        = torch.argmax(weighted, dim=1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(labels.numpy())
    return compute_metrics(np.array(y_true), np.array(y_pred))

In [19]:
@torch.no_grad()
def evaluate_gated_moe(model: GatedMoE, loader: DataLoader) -> Dict[str, float]:
    model.eval()
    y_true, y_pred = [], []

    for images, labels in loader:
        images    = images.to(device, non_blocking=True)
        logits, _ = model(images)
        preds     = torch.argmax(logits, dim=1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(labels.numpy())

    return compute_metrics(np.array(y_true), np.array(y_pred))

In [20]:
trained_experts: Dict[str, ExpertModel] = {}
expert_histories: Dict[str, Dict] = {}

for gen in CONFIG['generators']:
    print('\n' + '='*60)
    print(f'Training Expert: {gen}')
    print('='*60)

    # Resume from checkpoint if exists — skip training
    safe_name = gen.replace('/', '_')
    ckpt_path = os.path.join(CONFIG['checkpoint_dir'], f'expert_{safe_name}_best.pt')

    expert = build_expert(gen)

    if os.path.exists(ckpt_path):
        expert.load_state_dict(torch.load(ckpt_path, map_location=device))
        print(f'[RESUME] Loaded from checkpoint ✓')
        trained_experts[gen] = expert
        expert_histories[gen] = None  # no history when resumed
    else:
        print('[TRAIN] No checkpoint found — training from scratch')
        train_ds = AIGeneratorDataset(CONFIG['data_root'], gen, 'train', get_transforms('train'))
        val_ds   = AIGeneratorDataset(CONFIG['data_root'], gen, 'val',   get_transforms('val'))
        train_loader = make_loader(train_ds, shuffle=True,  drop_last=True)
        val_loader   = make_loader(val_ds,   shuffle=False, drop_last=False)
        history = train_expert(expert, train_loader, val_loader, gen)
        trained_experts[gen]  = expert
        expert_histories[gen] = history

print(f'\nAll {len(trained_experts)} experts ready.')


Training Expert: BigGan/imagenet_ai_0419_biggan
Expert [BigGan/imagenet_ai_0419_biggan]: 23.5M params
[RESUME] Loaded from checkpoint ✓

Training Expert: Glide/imagenet_glide
Expert [Glide/imagenet_glide]: 23.5M params
[RESUME] Loaded from checkpoint ✓

Training Expert: vqdm/imagenet_ai_0419_vqdm
Expert [vqdm/imagenet_ai_0419_vqdm]: 23.5M params
[RESUME] Loaded from checkpoint ✓

Training Expert: stable_diffusion/imagenet_ai_0419_sdv4
Expert [stable_diffusion/imagenet_ai_0419_sdv4]: 23.5M params
[RESUME] Loaded from checkpoint ✓

All 4 experts ready.


In [21]:
for expert in trained_experts.values():
    for param in expert.parameters():
        param.requires_grad = False

# Verify
all_frozen = all(
    not p.requires_grad
    for expert in trained_experts.values()
    for p in expert.parameters()
)
assert all_frozen, '❌ Some experts still have requires_grad=True!'
print('All expert parameters frozen ✓')

All expert parameters frozen ✓


In [22]:
print('\n' + '='*60)
print('Training Gating Network (router only)')
print('='*60)

# Verify experts are frozen before training gating network
all_frozen = all(
    not p.requires_grad
    for expert in trained_experts.values()
    for p in expert.parameters()
)
assert all_frozen, '❌ Experts not frozen! Re-run freeze cell first.'
print('Experts frozen: ✓')

gating_ckpt = os.path.join(CONFIG['checkpoint_dir'], 'gated_best.pt')

gating_net = GatingNetwork(n_experts=len(CONFIG['generators'])).to(device)
gating_net = gating_net.to(memory_format=torch.channels_last)

trainable = sum(p.numel() for p in gating_net.parameters() if p.requires_grad)
print(f'Trainable params in GatingNetwork: {trainable:,}')

if os.path.exists(gating_ckpt):
    gating_net.load_state_dict(torch.load(gating_ckpt, map_location=device))
    gated_history = None
    print('[RESUME] Gating network loaded from checkpoint ✓')
else:
    print('[TRAIN] No checkpoint found — training from scratch')
    gate_train_ds     = FullDataset(CONFIG['data_root'], CONFIG['generators'], 'train', get_transforms('train'))
    gate_val_ds       = FullDataset(CONFIG['data_root'], CONFIG['generators'], 'val',   get_transforms('val'))
    gate_train_loader = make_loader(gate_train_ds, shuffle=True,  drop_last=True)
    gate_val_loader   = make_loader(gate_val_ds,   shuffle=False, drop_last=False)
    gated_history     = train_gated_model(
        gating_net, gate_train_loader, gate_val_loader,
        ckpt_name='gated_best.pt')

# Sanitize keys — '/' in expert keys breaks nn.ModuleDict
sanitized_experts = {k.replace('/', '_'): v for k, v in trained_experts.items()}
gated_moe = GatedMoE(gating_net, sanitized_experts).to(device)
print('\nGatedMoE built ✓')


Training Gating Network (router only)
Experts frozen: ✓
Trainable params in GatingNetwork: 4,291,332
[RESUME] Gating network loaded from checkpoint ✓

GatedMoE built ✓


In [23]:
@torch.no_grad()
def evaluate_average_ensemble(experts, loader):
    for m in experts.values():
        m.eval()

    y_true, y_pred = [], []

    for images, labels in loader:
        images     = images.to(device, non_blocking=True)
        avg_logits = None

        for m in experts.values():
            logits     = m(images)
            avg_logits = logits if avg_logits is None else avg_logits + logits

        avg_logits /= len(experts)
        preds = torch.argmax(avg_logits, dim=1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(labels.numpy())

    return compute_metrics(np.array(y_true), np.array(y_pred))

In [24]:
@torch.no_grad()
def evaluate_voting_ensemble(experts, loader):
    for m in experts.values():
        m.eval()

    y_true, y_pred = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        votes  = []

        for m in experts.values():
            preds = torch.argmax(m(images), dim=1)
            votes.append(preds)

        vote_stack = torch.stack(votes, dim=1)
        majority   = torch.mode(vote_stack, dim=1).values.cpu().numpy()
        y_pred.extend(majority)
        y_true.extend(labels.numpy())

    return compute_metrics(np.array(y_true), np.array(y_pred))

In [25]:
class UnifiedModel(nn.Module):
    def __init__(self):
        super().__init__()
        backbone       = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone  = nn.Sequential(*list(backbone.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(2048, 2)
        )
    def forward(self, x):
        feats = self.backbone(x).flatten(1)
        return self.classifier(feats)


def train_unified_model():
    model = UnifiedModel().to(device)
    print(f'Unified model: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

    # FIX 1: add checkpoint resume — skip training if already done
    ckpt_path = os.path.join(CONFIG['checkpoint_dir'], 'unified_best.pt')
    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        print('[RESUME] Unified model loaded from checkpoint ✓')
        return model

    train_ds     = FullDataset(CONFIG['data_root'], CONFIG['generators'], 'train', get_transforms('train'))
    val_ds       = FullDataset(CONFIG['data_root'], CONFIG['generators'], 'val',   get_transforms('val'))
    train_loader = make_loader(train_ds, shuffle=True)
    val_loader   = make_loader(val_ds,   shuffle=False)
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(model)
    scheduler = make_scheduler(optimizer, train_loader)
    scaler    = GradScaler("cuda", enabled=CONFIG['use_amp'])
    stopper   = EarlyStopping(patience=5)
    best_f1   = 0.0

    for epoch in range(CONFIG['num_epochs']):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.long().to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=CONFIG['use_amp']):
                logits = model(images)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += loss.item()

        metrics = evaluate_model(model, val_loader)
        print(f'[Unified] Epoch {epoch+1:02d} loss={running_loss/len(train_loader):.4f}', end=' ')
        print_metrics(metrics)

        # FIX 2: save checkpoint instead of in-memory best_state
        if metrics['f1'] > best_f1:
            best_f1 = metrics['f1']
            torch.save(model.state_dict(),
                os.path.join(CONFIG['checkpoint_dir'], 'unified_best.pt'))

        if stopper.step(metrics['f1']):
            break

    # FIX 3: load from checkpoint instead of in-memory best_state
    model.load_state_dict(torch.load(
        os.path.join(CONFIG['checkpoint_dir'], 'unified_best.pt'),
        map_location=device))
    print(f'[Unified] Best F1: {best_f1:.4f}')
    return model

In [26]:
print('\n' + '='*60)
print('Training Attention Gating Network (novelty)')
print('='*60)

# Sanitize keys — '/' breaks nn.ModuleDict
sanitized_experts = {k.replace('/', '_'): v for k, v in trained_experts.items()}

attn_gate        = AttentionGatingNetwork(n_experts=4, patch_size=23,
                                           embed_dim=128, num_heads=4,
                                           ff_dim=256).to(device)
attn_gate        = attn_gate.to(memory_format=torch.channels_last)

attn_gated_model = AttentionGatedMoE(attn_gate, sanitized_experts).to(device)

trainable = sum(p.numel() for p in attn_gated_model.parameters() if p.requires_grad)
print(f'Trainable params (attention gate only): {trainable:,}')

attn_ckpt = os.path.join(CONFIG['checkpoint_dir'], 'attn_gated_best.pt')

if os.path.exists(attn_ckpt):
    attn_gate.load_state_dict(torch.load(attn_ckpt, map_location=device))
    attn_gated_history = None
    print(f'[RESUME] Attention gating network loaded from checkpoint ✓')
else:
    print('[TRAIN] No checkpoint found — training from scratch')
    attn_train_ds     = FullDataset(CONFIG['data_root'], CONFIG['generators'], 'train', get_transforms('train'))
    attn_val_ds       = FullDataset(CONFIG['data_root'], CONFIG['generators'], 'val',   get_transforms('val'))
    attn_train_loader = make_loader(attn_train_ds, shuffle=True,  drop_last=True)
    attn_val_loader   = make_loader(attn_val_ds,   shuffle=False, drop_last=False)

    attn_gated_history = train_gated_model(
        attn_gated_model, attn_train_loader, attn_val_loader,
        ckpt_name='attn_gated_best.pt')

print('\nAttention Gated Network ready ✓')



Training Attention Gating Network (novelty)
Trainable params (attention gate only): 402,817
[TRAIN] No checkpoint found — training from scratch
[Full/train] 349995 total images
[Full/val] 75000 total images


/SASTRA-NEW-CLUSTER/apps/anaconda3/envs/LABENV/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


[Gated] Epoch 01/20 loss=0.0338 lr=0.001043 ES=0/5 952.3s   Acc=0.9918  Prec=0.9858  Rec=0.9981  F1=0.9919
[Gated] Epoch 02/20 loss=0.0298 lr=0.002800 ES=0/5 829.0s   Acc=0.9883  Prec=0.9781  Rec=0.9990  F1=0.9884
[Gated] Epoch 03/20 loss=0.0304 lr=0.005200 ES=1/5 1010.7s   Acc=0.9930  Prec=0.9889  Rec=0.9971  F1=0.9930
[Gated] Epoch 04/20 loss=0.0307 lr=0.007601 ES=0/5 984.2s   Acc=0.9935  Prec=0.9910  Rec=0.9961  F1=0.9935
[Gated] Epoch 05/20 loss=0.0291 lr=0.009357 ES=0/5 1067.2s   Acc=0.9941  Prec=0.9920  Rec=0.9962  F1=0.9941
[Gated] Epoch 06/20 loss=0.0262 lr=0.010000 ES=0/5 1048.5s   Acc=0.9934  Prec=0.9916  Rec=0.9951  F1=0.9934
[Gated] Epoch 07/20 loss=0.0246 lr=0.009875 ES=1/5 1106.1s   Acc=0.9931  Prec=0.9890  Rec=0.9974  F1=0.9932
[Gated] Epoch 08/20 loss=0.0232 lr=0.009505 ES=2/5 1028.2s   Acc=0.9932  Prec=0.9891  Rec=0.9975  F1=0.9933
[Gated] Epoch 09/20 loss=0.0222 lr=0.008909 ES=3/5 968.7s   Acc=0.9938  Prec=0.9920  Rec=0.9956  F1=0.9938
[Gated] Epoch 10/20 loss=0.0219 

In [29]:
attn_metrics = evaluate_gated_moe(attn_gated_model, val_loader)
print_metrics(attn_metrics, prefix='Attention GatedMoE')

Attention GatedMoE  Acc=0.9941  Prec=0.9920  Rec=0.9962  F1=0.9941


In [28]:
print('\nTraining Unified Model...')
unified_model = train_unified_model()

print('\nEvaluating all methods...')
val_ds     = FullDataset(CONFIG['data_root'], CONFIG['generators'], 'val', get_transforms('val'))
val_loader = make_loader(val_ds, shuffle=False)

avg_metrics     = evaluate_average_ensemble(trained_experts, val_loader)
vote_metrics    = evaluate_voting_ensemble(trained_experts, val_loader)
unified_metrics = evaluate_model(unified_model, val_loader)
gated_metrics   = evaluate_gated_moe(gated_moe, val_loader)          # ADDED

print('\n===== FINAL RESULTS =====')
print_metrics(avg_metrics,     prefix='Average Ensemble  ')
print_metrics(vote_metrics,    prefix='Voting Ensemble   ')
print_metrics(unified_metrics, prefix='Unified Model     ')
print_metrics(gated_metrics,   prefix='Gated MoE         ') 


Training Unified Model...
Unified model: 23.5M params
[RESUME] Unified model loaded from checkpoint ✓

Evaluating all methods...
[Full/val] 75000 total images

===== FINAL RESULTS =====
Average Ensemble    Acc=0.9763  Prec=0.9847  Rec=0.9675  F1=0.9760
Voting Ensemble     Acc=0.7923  Prec=0.9889  Rec=0.5906  F1=0.7395
Unified Model       Acc=0.9982  Prec=0.9985  Rec=0.9978  F1=0.9982
Gated MoE           Acc=0.9828  Prec=0.9771  Rec=0.9887  F1=0.9829


In [30]:
def plot_training_history(history: dict, title: str, save_path: str = None):
    epochs = list(range(1, len(history['train_loss']) + 1))

    fig = plt.figure(figsize=(18, 10))
    fig.suptitle(f"Training Dashboard — {title}", fontsize=14, fontweight='bold', y=1.01)
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(epochs, history['train_loss'], color='#E24B4A', linewidth=2, marker='o', markersize=3)
    ax1.set_title("Training Loss", fontweight='bold')
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.grid(True, alpha=0.3)
    best_epoch = history['train_loss'].index(min(history['train_loss'])) + 1
    ax1.axvline(best_epoch, color='gray', linestyle='--', alpha=0.5, label=f'Min epoch {best_epoch}')
    ax1.legend(fontsize=8)

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(epochs, history['val_acc'],  label='Accuracy',  color='#378ADD', linewidth=2, marker='o', markersize=3)
    ax2.plot(epochs, history['val_prec'], label='Precision', color='#1D9E75', linewidth=2, marker='s', markersize=3)
    ax2.plot(epochs, history['val_rec'],  label='Recall',    color='#EF9F27', linewidth=2, marker='^', markersize=3)
    ax2.plot(epochs, history['val_f1'],   label='F1',        color='#D85A30', linewidth=2, marker='D', markersize=3)
    ax2.set_title("Validation Metrics", fontweight='bold')
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Score")
    ax2.set_ylim([max(0, min(history['val_f1']) - 0.05), 1.02])
    ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

    ax3 = fig.add_subplot(gs[0, 2])
    ax3.plot(epochs, history['lr'], color='#7F77DD', linewidth=2, marker='o', markersize=3)
    ax3.set_title("Learning Rate (OneCycleLR)", fontweight='bold')
    ax3.set_xlabel("Epoch"); ax3.set_ylabel("LR")
    ax3.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
    ax3.grid(True, alpha=0.3)

    ax4 = fig.add_subplot(gs[1, 0])
    ax4.plot(epochs, history['val_f1'], color='#D85A30', linewidth=2, marker='o', markersize=3)
    best_f1_epoch = history['val_f1'].index(max(history['val_f1'])) + 1
    best_f1_val   = max(history['val_f1'])
    ax4.axvline(best_f1_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best F1={best_f1_val:.4f} @ ep{best_f1_epoch}')
    ax4.scatter([best_f1_epoch], [best_f1_val], color='green', s=80, zorder=5)
    ax4.set_title("F1 Score + Best Epoch", fontweight='bold')
    ax4.set_xlabel("Epoch"); ax4.set_ylabel("F1")
    ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3)

    ax5 = fig.add_subplot(gs[1, 1])
    ax5.bar(epochs, history['early_stop_counter'],
            color=['#E24B4A' if c > 0 else '#9FE1CB' for c in history['early_stop_counter']],
            alpha=0.8, edgecolor='white')
    ax5.axhline(5, color='red', linestyle='--', linewidth=1.5, label='Patience limit (5)')
    ax5.set_title("Early Stopping Counter", fontweight='bold')
    ax5.set_xlabel("Epoch"); ax5.set_ylabel("Counter")
    ax5.set_yticks(range(0, 7))
    ax5.legend(fontsize=8); ax5.grid(True, alpha=0.3, axis='y')

    ax6 = fig.add_subplot(gs[1, 2])
    color_loss = '#E24B4A'
    color_f1   = '#378ADD'
    ax6.plot(epochs, history['train_loss'], color=color_loss, linewidth=2, label='Train Loss')
    ax6.set_xlabel("Epoch")
    ax6.set_ylabel("Loss", color=color_loss)
    ax6.tick_params(axis='y', labelcolor=color_loss)
    ax6_twin = ax6.twinx()
    ax6_twin.plot(epochs, history['val_f1'], color=color_f1, linewidth=2, linestyle='--', label='Val F1')
    ax6_twin.set_ylabel("F1", color=color_f1)
    ax6_twin.tick_params(axis='y', labelcolor=color_f1)
    ax6.set_title("Loss vs F1", fontweight='bold')
    lines1, labels1 = ax6.get_legend_handles_labels()
    lines2, labels2 = ax6_twin.get_legend_handles_labels()
    ax6.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='center right')
    ax6.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved → {save_path}")
    plt.show()
    plt.close()


# ── Plot all expert histories ────────────────────────────────
for gen, hist in expert_histories.items():
    if hist is None:
        print(f'No training history for {gen} (loaded from checkpoint)')
        continue
    safe_name = gen.replace('/', '_')
    plot_training_history(
        hist,
        title=gen,
        save_path=os.path.join(CONFIG['results_dir'], f'expert_{safe_name}_training.png')
    )

# ── CHANGED: plot gated + attention histories with None guard ─
if gated_history is not None:
    plot_training_history(gated_history, title="Gated Network (MLP)",
        save_path=os.path.join(CONFIG['results_dir'], 'gated_training.png'))

if attn_gated_history is not None:
    plot_training_history(attn_gated_history, title="Attention Gated Network",
        save_path=os.path.join(CONFIG['results_dir'], 'attn_gated_training.png'))

No training history for BigGan/imagenet_ai_0419_biggan (loaded from checkpoint)
No training history for Glide/imagenet_glide (loaded from checkpoint)
No training history for vqdm/imagenet_ai_0419_vqdm (loaded from checkpoint)
No training history for stable_diffusion/imagenet_ai_0419_sdv4 (loaded from checkpoint)


/tmp/ipykernel_3976405/1020548176.py:71: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Saved → ./results/attn_gated_training.png


In [31]:
import subprocess
subprocess.run(['pip', 'install', 'pandas', '--quiet'])
import pandas as pd

GENERATOR_NAMES = ['BigGAN', 'GLIDE', 'VQDM', 'SDiffusion']
GENERATOR_KEYS  = CONFIG['generators']

# ── Helper: evaluate any model on a specific generator's val set ──
def evaluate_on_generator(model, gen_key, eval_fn=evaluate_model):
    ds     = AIGeneratorDataset(CONFIG['data_root'], gen_key, 'val', get_transforms('val'))
    loader = make_loader(ds, shuffle=False, drop_last=False)
    return eval_fn(model, loader)

# ── Helper: print table like paper ──────────────────────────────────
def print_table(title, results_per_generator):
    print(f'\n{title}')
    print(f'{"Metric":<12}' + ''.join(f'{n:>12}' for n in GENERATOR_NAMES))
    print('-' * (12 + 12 * len(GENERATOR_NAMES)))
    for metric, label in [('accuracy',  'Accuracy'),
                           ('precision', 'Precision'),
                           ('recall',    'Recall'),
                           ('f1',        'F1-Score')]:
        row = f'{label:<12}'
        for g in GENERATOR_KEYS:
            row += f'{results_per_generator[g][metric]*100:>11.2f}%'
        print(row)

# ── Helper: build pandas DataFrame for saving ───────────────────────
def build_df(results_per_generator):
    rows = {}
    for metric, label in [('accuracy',  'Accuracy'),
                           ('precision', 'Precision'),
                           ('recall',    'Recall'),
                           ('f1',        'F1-Score')]:
        rows[label] = {
            GENERATOR_NAMES[i]: f"{results_per_generator[g][metric]*100:.2f}%"
            for i, g in enumerate(GENERATOR_KEYS)
        }
    return pd.DataFrame(rows).T

# ════════════════════════════════════════════════════════════════════
# TABLES 3–6: Expert models tested on ALL generators
# ════════════════════════════════════════════════════════════════════
expert_cross_results = {}

for i, (exp_key, expert) in enumerate(trained_experts.items()):
    results = {}
    for gen_key in GENERATOR_KEYS:
        results[gen_key] = evaluate_on_generator(expert, gen_key)
    expert_cross_results[exp_key] = results

    table_num = i + 3
    title     = f'TABLE {table_num}. Testing of models trained on {GENERATOR_NAMES[i]}.'
    print_table(title, results)
    build_df(results).to_csv(
        os.path.join(CONFIG['results_dir'],
                     f'table_{table_num}_expert_{GENERATOR_NAMES[i]}.csv'))

# ════════════════════════════════════════════════════════════════════
# TABLE 7: Averaging Ensemble
# ════════════════════════════════════════════════════════════════════
avg_results = {}
for gen_key in GENERATOR_KEYS:
    ds     = AIGeneratorDataset(CONFIG['data_root'], gen_key, 'val', get_transforms('val'))
    loader = make_loader(ds, shuffle=False, drop_last=False)
    avg_results[gen_key] = evaluate_average_ensemble(trained_experts, loader)

print_table('TABLE 7. Testing of averaging ensemble model.', avg_results)
build_df(avg_results).to_csv(
    os.path.join(CONFIG['results_dir'], 'table_7_avg_ensemble.csv'))

# ════════════════════════════════════════════════════════════════════
# TABLE 8: Voting Ensemble
# ════════════════════════════════════════════════════════════════════
vote_results = {}
for gen_key in GENERATOR_KEYS:
    ds     = AIGeneratorDataset(CONFIG['data_root'], gen_key, 'val', get_transforms('val'))
    loader = make_loader(ds, shuffle=False, drop_last=False)
    vote_results[gen_key] = evaluate_voting_ensemble(trained_experts, loader)

print_table('TABLE 8. Testing of voting ensemble model.', vote_results)
build_df(vote_results).to_csv(
    os.path.join(CONFIG['results_dir'], 'table_8_vote_ensemble.csv'))

# ════════════════════════════════════════════════════════════════════
# TABLE 9: Unified Model
# ════════════════════════════════════════════════════════════════════
unified_results = {}
for gen_key in GENERATOR_KEYS:
    unified_results[gen_key] = evaluate_on_generator(unified_model, gen_key)

print_table('TABLE 9. Testing of model with unified training.', unified_results)
build_df(unified_results).to_csv(
    os.path.join(CONFIG['results_dir'], 'table_9_unified.csv'))

# ════════════════════════════════════════════════════════════════════
# TABLE 10: Gated Network (MLP)
# ════════════════════════════════════════════════════════════════════
gated_results = {}
for gen_key in GENERATOR_KEYS:
    ds     = AIGeneratorDataset(CONFIG['data_root'], gen_key, 'val', get_transforms('val'))
    loader = make_loader(ds, shuffle=False, drop_last=False)
    gated_results[gen_key] = evaluate_gated_moe(gated_moe, loader)

print_table('TABLE 10. Testing of gated network model.', gated_results)
build_df(gated_results).to_csv(
    os.path.join(CONFIG['results_dir'], 'table_10_gated.csv'))

# ════════════════════════════════════════════════════════════════════
# TABLE 11: Attention Gated Network (novelty)
# ════════════════════════════════════════════════════════════════════
attn_results = {}
for gen_key in GENERATOR_KEYS:
    ds     = AIGeneratorDataset(CONFIG['data_root'], gen_key, 'val', get_transforms('val'))
    loader = make_loader(ds, shuffle=False, drop_last=False)
    attn_results[gen_key] = evaluate_gated_moe(attn_gated_model, loader)

print_table('TABLE 11. Testing of attention gated network model.', attn_results)
build_df(attn_results).to_csv(
    os.path.join(CONFIG['results_dir'], 'table_11_attn_gated.csv'))

# ════════════════════════════════════════════════════════════════════
# TABLE 16: Accuracy comparison among all methods
# ════════════════════════════════════════════════════════════════════
all_methods = {
    **{f'Trained on {GENERATOR_NAMES[i]}': expert_cross_results[k]
       for i, k in enumerate(GENERATOR_KEYS)},
    'Averaging Ensemble'    : avg_results,
    'Voting Ensemble'       : vote_results,
    'Unified Training'      : unified_results,
    'Gated Network (MLP)'   : gated_results,
    'Attention Gated Network': attn_results,
}

method_accs = {}
for method_name, results in all_methods.items():
    accs = [results[g]['accuracy'] for g in GENERATOR_KEYS]
    method_accs[method_name] = accs + [sum(accs) / len(accs)]

cols = GENERATOR_NAMES + ['Average']

best_idx   = {}
second_idx = {}
for col_i in range(len(cols)):
    col_vals   = [(name, vals[col_i]) for name, vals in method_accs.items()]
    sorted_col = sorted(col_vals, key=lambda x: x[1], reverse=True)
    best_idx[col_i]   = sorted_col[0][0]
    second_idx[col_i] = sorted_col[1][0]

BOLD      = '\033[1m'
UNDERLINE = '\033[4m'
RESET     = '\033[0m'

print('\n\nTABLE 16. Accuracy comparison among methods.')
header = f'{"Model":<25}' + ''.join(f'{n:>13}' for n in cols)
print(header)
print('-' * (25 + 13 * len(cols)))

for method_name, vals in method_accs.items():
    row = f'{method_name:<25}'
    for col_i, val in enumerate(vals):
        cell = f'{val*100:>11.2f}%'
        if method_name == best_idx[col_i]:
            cell = BOLD + cell + RESET
        elif method_name == second_idx[col_i]:
            cell = UNDERLINE + cell + RESET
        row += f'  {cell}'
    print(row)

rows_16 = []
for method_name, vals in method_accs.items():
    row_dict = {'Model': method_name}
    for col_name, val in zip(cols, vals):
        row_dict[col_name] = f'{val*100:.2f}%'
    rows_16.append(row_dict)

df_16 = pd.DataFrame(rows_16).set_index('Model')
csv_path_16 = os.path.join(CONFIG['results_dir'], 'table_16_accuracy_comparison.csv')
df_16.to_csv(csv_path_16)
print(f'\nAll tables saved to {CONFIG["results_dir"]}/')


[BigGan/imagenet_ai_0419_biggan/val] 18750 images | real=9375, fake=9375
[Glide/imagenet_glide/val] 18750 images | real=9375, fake=9375
[vqdm/imagenet_ai_0419_vqdm/val] 18750 images | real=9375, fake=9375
[stable_diffusion/imagenet_ai_0419_sdv4/val] 18750 images | real=9375, fake=9375

TABLE 3. Testing of models trained on BigGAN.
Metric            BigGAN       GLIDE        VQDM  SDiffusion
------------------------------------------------------------
Accuracy          98.68%      97.45%      71.27%      50.07%
Precision         97.43%      97.37%      94.04%      51.21%
Recall           100.00%      97.54%      45.43%       2.94%
F1-Score          98.70%      97.45%      61.26%       5.57%

[BigGan/imagenet_ai_0419_biggan/val] 18750 images | real=9375, fake=9375
[Glide/imagenet_glide/val] 18750 images | real=9375, fake=9375
[vqdm/imagenet_ai_0419_vqdm/val] 18750 images | real=9375, fake=9375
[stable_diffusion/imagenet_ai_0419_sdv4/val] 18750 images | real=9375, fake=9375

TABLE 4. Tes